# TikiTorch — deep learning without a single multiply

**TikiTorch** is a PyTorch-shaped deep-learning library whose entire learning
loop — forward, backward, *and* the optimizer — runs in the **tropical
semiring**: the only primitive operations are **compare, add, and bit-shift**.
There are **no multiplications anywhere**, which is exactly what lets a whole
training step map onto FPGA fabric with **zero DSP blocks** (zero hardware
multipliers).

In the tropical semiring, "multiply" becomes ordinary **addition** and "add"
becomes a **min** or **max** (a comparison). A linear layer
`y = Wx` becomes a *morphological* operation `y_j = min_i (W_ji + x_i)` (erosion)
or `max_i (W_ji + x_i)` (dilation). Stack those, add a tropical attention or a
morphological convolution, train with a sign-based optimizer, and you have a deep
net that never multiplies.

This notebook is a 10-minute tour. The others go deeper:
- **02_tropical_math** — the semiring, dilation/erosion, the soft→hard curriculum.
- **03_vision_cnn** — a fully multiplier-free CNN on MNIST.
- **04_sequence** — multiplier-free language modeling + the closure-faithful loss.


In [ ]:
# --- Setup: install TikiTorch (run once) ---
# TikiTorch is research code, not on PyPI. In Colab, install from the repo.
# Replace REPO_URL with your TikiTorch checkout (it contains tikitorch/ + data/).
try:
    import tikitorch  # already installed?
except ImportError:
    REPO_URL = "https://github.com/your-org/tikitorch.git"  # <-- set me
    import subprocess, sys, os
    if not os.path.exists("tikitorch_repo"):
        subprocess.run(["git", "clone", REPO_URL, "tikitorch_repo"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "tikitorch_repo/tikitorch"], check=False)
    import tikitorch
import torch
print("TikiTorch", getattr(tikitorch, "__version__", "?"), "| torch", torch.__version__)


## A complete training step in ten lines

TikiTorch mirrors the PyTorch namespace: `tt.nn`, `tt.optim`, `tt.models`,
`tt.functional`. Here is a morphological state-space LM trained one step with a
strictly multiplier-free optimizer.


In [ ]:
import torch
import tikitorch as tt
import tikitorch.optim as optim

model = tt.models.MorphSSM(d_model=128, n_layers=2, task_type="lm", vocab_size=128)
opt = optim.PureTropicalSGD(model.parameters(), lr_shift=10)  # lr = 2^-10, a bit-shift

x = torch.randint(0, 128, (8, 64))   # (batch, seq)
y = torch.randint(0, 128, (8, 64))
loss = model.compute_loss(x, y)      # cross-entropy
loss.backward()                       # gradients route through argmin/argmax
opt.step(); opt.zero_grad()
print("loss:", round(loss.item(), 4))


## The zero-multiply claim is *checkable*

The headline isn't a vibe — it's mechanically verifiable. `assert_closed` walks a
model and raises if it finds **any** standard `torch.nn` module that contains a
hardware multiply or divide (`nn.Linear`, `nn.Conv2d`, `nn.LayerNorm`, the cuDNN
RNNs, ...). A TikiTorch model built from tropical primitives passes.


In [ ]:
from tikitorch.audit import assert_closed, find_closure_violations

cnn = tt.models.TropicalCNN(in_channels=1, num_classes=10)   # morphological CNN
assert_closed(cnn)
print("TropicalCNN is fully multiplier-free  ✓  (params:", cnn.param_count, ")")

# A standard nn.Linear head is a *declared* DSP boundary — the checker names it:
boundary = tt.models.TropicalCNN(head_type="linear")
print("violations in linear-head variant:", find_closure_violations(boundary))


## Why this matters: it runs on silicon at single-digit millijoules

Because the loop never multiplies, the same computation synthesizes to **zero DSP
blocks** on an FPGA. TikiTorch ships the HLS kernels; the architectures here were
trained **end-to-end on AWS F2 silicon** (Virtex UltraScale+ VU47P):

| Build | Throughput | Energy | DSP |
|---|---|---|---|
| 2-layer tropical LM (char) | ~772 tok/s | ~9.0 mJ/token | **0** |
| MorphSSM ECG (clinical) | ~1207 tok/s | ~5.8 mJ/token | **0** |

The accuracy cost of this *closure* is real and measured (a few points on
classification; larger on hard char-LM) — and TikiTorch gives you the knobs (the
soft→hard `tau` curriculum, the loss/optimizer choices) to trade it back. That's
the subject of the other notebooks.

**The map:**

| namespace | what |
|---|---|
| `tt.functional` | stateless tropical ops + autograd (linear, dilation, conv2d, attention, losses) |
| `tt.nn` | layer modules (TropicalLinear, TropicalConv2d, TropicalAttention, TropicalLayerNorm, ...) |
| `tt.optim` | multiplier-free optimizers (SignSGD, TropicalLion, PureTropicalSGD) |
| `tt.models` | ready architectures (TropicalMLP/CNN, MorphSSM, TropicalTransformer, MoTE) |
| `tt.audit` | `assert_closed` — verify a model is multiplier-free |
